# Phase 1 — Sparse Retriever (BM25)
This notebook builds a BM25 index (Python-only, `rank_bm25`) over a chosen subset, retrieves top-k per query, saves a TREC run file, and evaluates nDCG@10 & MRR@10.

> Matches SOP Step 3 & Step 5 (Sparse Retriever + Baseline Evaluation).

In [ ]:
# Optional: install
# !pip install -r requirements.txt

In [11]:
from __future__ import annotations
import json, math
from pathlib import Path
from typing import List, Tuple, Dict
import numpy as np
from tqdm import tqdm
from rank_bm25 import BM25Okapi

WORK_DIR = Path("./work")
# Example: choose the dataset & subset you generated in Notebook 2
SUBSET_DIR = WORK_DIR / "subsets" / "beir_trec-covid" / "subset_3"

#SUBSET_DIR = WORK_DIR / "subsets" / "beir_trec-covid_dev" / "subset_1"
TOP_K = 100

def read_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)

def tokenize_simple(text: str):
    import re
    return re.findall(r"[a-z0-9]+", text.lower())

def trec_run_write(path: Path, records, system_name="bm25"):
    by_qid = {}
    for qid, docid, score in records:
        by_qid.setdefault(qid, []).append((docid, float(score)))
    with path.open("w", encoding="utf-8") as f:
        for qid, pairs in by_qid.items():
            pairs.sort(key=lambda x: x[1], reverse=True)
            for rank, (docid, score) in enumerate(pairs[:1000], start=1):
                f.write(f"{qid} Q0 {docid} {rank} {score:.6f} {system_name}\n")

In [12]:
# Build BM25 index
corpus = list(read_jsonl(SUBSET_DIR / "corpus.jsonl"))
docids = [r["doc_id"] for r in corpus]
tokenized = [tokenize_simple(r["text"]) for r in corpus]
bm25 = BM25Okapi(tokenized)

In [13]:
# Retrieve
queries = list(read_jsonl(SUBSET_DIR / "queries.jsonl"))
run_path = SUBSET_DIR / "run_bm25.trec"

def gen_records():
    for q in tqdm(queries, desc="BM25 retrieve"):
        qtok = tokenize_simple(q["text"])
        scores = bm25.get_scores(qtok)
        idxs = np.argsort(scores)[::-1][:TOP_K]
        for i in idxs:
            yield (q["qid"], docids[i], float(scores[i]))

trec_run_write(run_path, gen_records(), system_name="bm25")
print("Run written:", run_path)

BM25 retrieve: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:03<00:00,  5.12it/s]

Run written: work/subsets/beir_trec-covid/subset_3/run_bm25.trec


In [14]:
# Evaluation: nDCG@10, MRR@10
def read_run(path: Path) -> Dict[str, List[Tuple[str, float]]]:
    per_q = {}
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 6: 
                continue
            qid, _, docid, _, score, _sys = parts[:6]
            per_q.setdefault(qid, []).append((docid, float(score)))
    for qid in per_q:
        per_q[qid].sort(key=lambda x: x[1], reverse=True)
    return per_q

qrels = {}
for r in read_jsonl(SUBSET_DIR / "qrels.jsonl"):
    qrels.setdefault(str(r["qid"]), {})[str(r["doc_id"])] = int(r["rel"])

def compute_mrr_at_k(run, qrels, k=10):
    mrrs = []
    for qid, ranked in run.items():
        relset = qrels.get(qid, {})
        rr = 0.0
        for i, (docid, _) in enumerate(ranked[:k], start=1):
            if relset.get(docid, 0) > 0:
                rr = 1.0 / i
                break
        mrrs.append(rr)
    import numpy as np
    return float(np.mean(mrrs)) if mrrs else 0.0
from pathlib import Path
import json

def doc_set(p):
    return {json.loads(x)['doc_id'] for x in open(p/'corpus.jsonl', 'r', encoding='utf-8')}
def q_set(p):
    return {json.loads(x)['qid'] for x in open(p/'queries.jsonl', 'r', encoding='utf-8')}

s1 = Path('./work/subsets/beir_trec-covid/subset_1')
s2 = Path('./work/subsets/beir_trec-covid/subset_2')

d1, d2 = doc_set(s1), doc_set(s2)
q1, q2 = q_set(s1), q_set(s2)

print(len(d1), len(d2), 'same_docs?', d1 == d2)
print(len(q1), len(q2), 'same_queries?', q1 == q2)

def compute_ndcg_at_k(run, qrels, k=10):
    import math
    def dcg(rels):
        return sum((rel / math.log2(i + 2)) for i, rel in enumerate(rels))
    ndcgs = []
    for qid, ranked in run.items():
        rels = [1 if qrels.get(qid, {}).get(docid, 0) > 0 else 0 for docid, _ in ranked[:k]]
        idcg = dcg(sorted(rels, reverse=True))
        nd = (dcg(rels) / idcg) if idcg > 0 else 0.0
        ndcgs.append(nd)
    import numpy as np
    return float(np.mean(ndcgs)) if ndcgs else 0.0

run = read_run(run_path)
print({"nDCG@10": compute_ndcg_at_k(run, qrels, 10), "MRR@10": compute_mrr_at_k(run, qrels, 10)})

55912 55931 same_docs? False
17 17 same_queries? False
{'nDCG@10': 0.9092289731645428, 'MRR@10': 0.875}


In [15]:
from pathlib import Path
import json

def doc_set(p):
    return {json.loads(x)['doc_id'] for x in open(p/'corpus.jsonl', 'r', encoding='utf-8')}
def q_set(p):
    return {json.loads(x)['qid'] for x in open(p/'queries.jsonl', 'r', encoding='utf-8')}

s1 = Path('./work/subsets/beir_trec-covid/subset_1')
s2 = Path('./work/subsets/beir_trec-covid/subset_2')

d1, d2 = doc_set(s1), doc_set(s2)
q1, q2 = q_set(s1), q_set(s2)

print(len(d1), len(d2), 'same_docs?', d1 == d2)
print(len(q1), len(q2), 'same_queries?', q1 == q2)


55912 55931 same_docs? False
17 17 same_queries? False
